TOKEN= SENH DO BOT PEGAR NO TELEGRAM



In [ ]:
import speech_recognition as sr
from ultralytics import YOLO
from telegram import Update  # codigo "conversar" com o tlegram
from telegram.ext import Application, MessageHandler, filters, ContextTypes


class BotTelegram:

    def __init__(self, token):
        # encapsulamento do token
        self.__token = token


# classe filha Imagem
class BotImagem(BotTelegram):

    def __init__(self, token):
        super().__init__(token)
        self.modelo = YOLO("yolov8n.pt")

    # usa do yolo pra avaliar a imagem
    def processar(self, imagem):
        resultados = self.modelo(imagem)

        # lista onde o yolo vai guardar o que detectou na imagem
        classes = []

        # Percorre cada objeto detectado e defini o nome de cada um(antes salvo como numero DENTRO DE BOX1,BOX2...), mandando pra lista classes
        for i in resultados:
            for box in i.boxes:
                cls_id = int(box.cls)
                nome = self.modelo.names[cls_id]
                # manda pra lista
                classes.append(nome)
        return classes


# Classe filha audio
class BotAudio(BotTelegram):

    def __init__(self, token):  # atributos da classe filha
        super().__init__(
            token
        )  # super com atribut. da classe mãe (HERANÇA VAI PROCESSAR O QUE FOI HERDADO)
        self.reconhecedor = (
            sr.Recognizer()
        )  # o atributo está sendo criado recebendo uma ferramenta da biblioteca

    # motodo audio
    def processar(self, audio):
        try:
            with sr.AudioFile(
                audio
            ) as fonte:  # with abre o arquivo e Sr.audioFile lê o audio
                dados_audio = self.reconhecedor.record(
                    fonte
                )  # chama a ferramenta pra ler o audio
                texto_extraido = self.reconhecedor.recognize_google(
                    dados_audio, language="pt-BR"
                )  # cnoverte em texto
                return f"Áudio convertido em texto: '{texto_extraido}'"

        except sr.UnknownValueError:  # em caso de erro imprime aviso
            return "Não foi possível entender o áudio..."
        except sr.RequestError as e:
            return f"Erro ao conectar ao serviço..."


class BotTexto(BotTelegram):
    # Ajustado apenas para receber o parâmetro do texto enviado

    def processar(self, texto):
        return f"Processando texto: {texto}"


class BotVideo(BotTelegram):

    def __init__(self, token):
        super().__init__(token)
        self.modelo = YOLO("yolov8n.pt")

    def processar(self, video_path):
        resultados = self.modelo(video_path, stream=True)

        classes_detectadas = set()
        # Lê os primeiros frames para não travar o bot coletando objetos
        for i, resultado in enumerate(resultados):
            if i > 30:  # Limita a análise aos primeiros frames para ser rápido
                break
            for box in resultado.boxes:
                cls_id = int(box.cls)
                classes_detectadas.add(self.modelo.names[cls_id])

        return list(classes_detectadas)


class BotDocumento(BotTelegram):

    def processar(self, nome_arquivo):
        # Lógica para processar o documento enviado
        return f"Documento '{nome_arquivo}' recebido e arquivado com sucesso!"


# determinar o tipo de arquivo
def reconhecerFormato(update, token):
    if update.message.photo:
        return BotImagem(token)

    elif update.message.voice or update.message.audio:
        return BotAudio(token)

    elif update.message.video or update.message.video_note:
        return BotVideo(token)

    elif update.message.document:
        return BotDocumento(token)

    elif update.message.text:
        return BotTexto(token)

    else:
        return None  # Caso seja outro tipo de mídia não mapeada


# Conexão da função reconhecerFormato com a rede do Telegram
async def lidar_com_mensagem(update: Update, context: ContextTypes.DEFAULT_TYPE):
    TOKEN = "SEU_TOKEN_AQUI"  # Mantenha o mesmo token configurado no main

    # Garante que a mensagem não está vazia
    if not update.message:
        return

    # Chama a função original para decidir qual classe instanciar
    bot = reconhecerFormato(update, TOKEN)

    if bot is None:
        await update.message.reply_text("Formato de mensagem não suportado.")
        return

    # Executa a lógica baseada na classe que retornou
    if isinstance(bot, BotImagem):
        arquivo = await update.message.photo[-1].get_file()
        await arquivo.download_to_drive("midia_bot.jpg")

        resposta = bot.processar("midia_bot.jpg")
        await update.message.reply_text(f"Objetos detectados na imagem: {resposta}")

    elif isinstance(bot, BotAudio):
        arquivo_audio = (
            update.message.voice
            if update.message.voice
            else update.message.audio
        )
        arquivo = await arquivo_audio.get_file()
        await arquivo.download_to_drive("midia_bot.wav")

        resposta = bot.processar("midia_bot.wav")
        await update.message.reply_text(resposta)

    elif isinstance(bot, BotVideo):
        arquivo_video = (
            update.message.video
            if update.message.video
            else update.message.video_note
        )
        arquivo = await arquivo_video.get_file()
        await arquivo.download_to_drive("midia_bot.mp4")

        resposta = bot.processar("midia_bot.mp4")
        await update.message.reply_text(f"Objetos detectados no vídeo: {resposta}")

    elif isinstance(bot, BotDocumento):
        arquivo = await update.message.document.get_file()
        nome_original = update.message.document.file_name
        await arquivo.download_to_drive(nome_original)

        resposta = bot.processar(nome_original)
        await update.message.reply_text(resposta)

    elif isinstance(bot, BotTexto):
        resposta = bot.processar(update.message.text)
        await update.message.reply_text(resposta)


def main():
    TOKEN = "SEU_TOKEN_AQUI"  # Coloque seu token gerado no BotFather aqui
    app = Application.builder().token(TOKEN).build()

    # O bot escuta as mensagens e passa para a lógica de formato
    app.add_handler(MessageHandler(filters.ALL, lidar_com_mensagem))

    print("Bot rodando...")
    app.run_polling()


if __name__ == "__main__":
    main()